In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-addon-utils
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 또는 그 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-addon-utils~=0.3.0
```
</details>

Qiskit 애드온 유틸리티 패키지는 하나 이상의 Qiskit 애드온을 포함하는 워크플로우를 보완하는 기능들의 모음입니다. 예를 들어, 이 패키지에는 해밀토니안 생성, Trotter 시간 진화 Circuit 생성, 양자 Circuit 슬라이싱 및 결합을 위한 함수들이 포함되어 있습니다.

## 설치

Qiskit 애드온 유틸리티를 설치하는 방법은 PyPI와 소스 빌드 두 가지가 있습니다. 패키지 의존성 간의 분리를 보장하기 위해 [가상 환경](https://docs.python.org/3.10/tutorial/venv.html)에 이 패키지들을 설치하는 것을 권장합니다.

### PyPI에서 설치

Qiskit 애드온 유틸리티 패키지를 설치하는 가장 간단한 방법은 PyPI를 통한 것입니다.

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit import QuantumCircuit
from qiskit_addon_utils.coloring import auto_color_edges
from qiskit_addon_utils.slicing import combine_slices, slice_by_depth
from collections import defaultdict

backend = FakeSherbrooke()
coupling_map = backend.coupling_map

# Create naive circuit
circuit = QuantumCircuit(backend.num_qubits)
for edge in coupling_map.graph.edge_list():
    circuit.cz(edge[0], edge[1])


# Color the edges of the coupling map
coloring = auto_color_edges(coupling_map)
circuit_with_coloring = QuantumCircuit(backend.num_qubits)

# Make a reverse coloring dict in order to make the circuit
color_to_edge = defaultdict(list)
for edge, color in coloring.items():
    color_to_edge[color].append(edge)

# Place edges in order of color
for edges in color_to_edge.values():
    for edge in edges:
        circuit_with_coloring.cz(edge[0], edge[1])

print(f"The circuit without using edge coloring has depth: {circuit.depth()}")
print(
    f"The circuit using edge coloring has depth: {circuit_with_coloring.depth()}"
)

The circuit without using edge coloring has depth: 37
The circuit using edge coloring has depth: 3


### 소스에서 설치
<details>
<summary>
이 패키지를 수동으로 설치하는 방법을 읽으려면 여기를 클릭하세요.
</summary>

이 패키지에 기여하거나 직접 설치하려면, 먼저 저장소를 클론하세요:

In [2]:
import numpy as np
from qiskit import QuantumCircuit

num_qubits = 9
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg" alt="Output of the previous code cell" />

그리고 `pip`을 통해 패키지를 설치하세요. 패키지 저장소에 있는 튜토리얼을 실행할 계획이라면 노트북 의존성도 함께 설치하세요. 저장소에서 개발할 계획이라면 `dev` 의존성을 설치하세요.

In [3]:
# Slice circuit into partitions of depth 1
slices = slice_by_depth(qc, 1)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/6d824d97-edc6-4c88-bcfa-428731393f83-0.svg" alt="Output of the previous code cell" />

</details>

## 유틸리티 시작하기
`qiskit-addon-utils` 패키지 내에는 여러 모듈이 있습니다. 양자 시스템 시뮬레이션을 위한 문제 생성 모듈, 양자 Circuit에서 Gate를 더 효율적으로 배치하기 위한 그래프 컬러링 모듈, 그리고 [연산자 역전파](/guides/qiskit-addons-obp)에 도움이 되는 Circuit 슬라이싱 모듈이 포함되어 있습니다. 다음 섹션에서 각 모듈을 요약하여 설명합니다. 패키지의 [API 문서](https://docs.quantum.ibm.com/api/qiskit-addon-utils)에도 유용한 정보가 포함되어 있습니다.

### 문제 생성
[`qiskit_addon_utils.problem_generators`](../api/qiskit-addon-utils/problem-generators) 모듈의 내용에는 다음이 포함됩니다:

- [`generate_xyz_hamiltonian()`](../api/qiskit-addon-utils/problem-generators#generate_xyz_hamiltonian) 함수: Ising형 XYZ 모델의 연결성 인식 `SparsePauliOp` 표현을 생성합니다:

$$H = \sum_{(j,k)\in E} \left(J_x X_jX_k + J_yY_jY_k + J_zZ_jZ_k\right) + \sum_{j\in V} \left(h_x X_j + h_y Y_j + h_z Z_j\right) $$
- [`generate_time_evolution_circuit()`](../api/qiskit-addon-utils/problem-generators#generate_time_evolution_circuit) 함수: 주어진 연산자의 시간 진화를 모델링하는 Circuit을 구성합니다.
- 다른 Pauli 문자열 순서를 열거하기 위한 세 가지 [`PauliOrderStrategy`](../api/qiskit-addon-utils/problem-generators#pauliorderstrategy) 객체. 이는 그래프 컬러링과 함께 사용할 때 특히 유용하며, `generate_xyz_hamiltonian()`과 `generate_time_evolution_circuit()` 함수 모두에서 인수로 사용할 수 있습니다.

### 그래프 컬러링
[`qiskit_addon_utils.coloring`](../api/qiskit-addon-utils/coloring) 모듈은 결합 맵의 에지에 색을 지정하고, 이 컬러링을 사용하여 양자 Circuit에서 Gate를 더 효율적으로 배치하는 데 사용됩니다. 에지 색상 결합 맵의 목적은 동일한 색상의 두 에지가 공통 노드를 공유하지 않도록 에지 색상 집합을 찾는 것입니다. QPU에서는 같은 색의 에지(Qubit 연결)를 따라 배치된 Gate들이 동시에 실행될 수 있으므로 Circuit이 더 빠르게 실행됩니다.

간단한 예제로, [`auto_color_edges()`](../api/qiskit-addon-utils/coloring#auto_color_edges) 함수를 사용하여 각 Qubit 연결을 따라 `CZGate`를 실행하는 단순한 Circuit에 대한 에지 컬러링을 생성할 수 있습니다. 아래 코드 스니펫은 `FakeSherbrooke` Backend의 결합 맵을 사용하여 이 단순한 Circuit을 만들고, `auto_color_edges()` 함수를 사용하여 더 효율적인 동등한 Circuit을 만듭니다.

In [4]:
from qiskit_addon_utils.slicing import slice_by_gate_types

slices = slice_by_gate_types(qc)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d011c56e-d741-491a-8867-54952cb7b757-0.svg" alt="Output of the previous code cell" />

If your workload is designed to exploit the physical qubit connectivity of the QPU it will be run on, you can create slices based on edge coloring. The code snippet below will assign a three-coloring to circuit edges and slice the circuit with respect to the edge coloring. (Note: this only affects non-local gates. Single-qubit gates will be sliced by gate type).

In [5]:
from qiskit_addon_utils.slicing import slice_by_coloring

# Assign a color to each set of connected qubits
coloring = {}
for i in range(num_qubits - 1):
    coloring[(i, i + 1)] = i % 3
coloring[(num_qubits - 1, 0)] = 2

# Create a circuit with operations added in order of color
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
edges = [
    edge for color in range(3) for edge in coloring if coloring[edge] == color
]
for edge in edges:
    qc.cx(edge[0], edge[1])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))

# Create slices by edge color
slices = slice_by_coloring(qc, coloring=coloring)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg" alt="Output of the previous code cell" />

### 슬라이싱
마지막으로, [`qiskit-addon-utils.slicing`](../api/qiskit-addon-utils/slicing) 모듈에는 모든 Qubit에 걸쳐 있는 [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit)의 시간적 파티션인 Circuit "슬라이스"를 만들기 위한 함수 및 Transpiler 패스가 포함되어 있습니다. 이 슬라이스들은 주로 [연산자 역전파](/guides/qiskit-addons-obp-get-started)에 사용됩니다. Circuit을 슬라이싱할 수 있는 주요 네 가지 방법은 Gate 유형, 깊이, 컬러링, 또는 [`Barrier`](../api/qiskit/circuit#barrier) 명령어에 의한 것입니다. 이러한 슬라이싱 함수의 출력은 [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit) 객체의 목록을 반환합니다. 슬라이싱된 Circuit은 [`combine_slices()`](../api/qiskit-addon-utils/slicing#combine_slices) 함수를 사용하여 다시 결합할 수도 있습니다. 자세한 정보는 모듈의 [API 참조](../api/qiskit-addon-utils/slicing)를 읽어보세요.

아래는 다음 Circuit을 사용하여 이러한 슬라이스를 만드는 몇 가지 예제입니다:

In [6]:
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qc.barrier()
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.barrier()
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/5040a678-9d5f-4c8b-8a92-7de04c3fcfda-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg)

연산자 역전파를 위해 Circuit의 구조를 활용할 명확한 방법이 없는 경우, Circuit을 주어진 깊이의 슬라이스로 파티션할 수 있습니다.

In [7]:
from qiskit_addon_utils.slicing import slice_by_barriers

slices = slice_by_barriers(qc)
slices[0].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/b1106c2f-711c-4d30-b91a-ea05f47598d8-0.svg" alt="Output of the previous code cell" />

In [8]:
slices[1].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/1afd2111-dd88-4e20-a137-f0c975e9d1bb-0.svg" alt="Output of the previous code cell" />

In [9]:
slices[2].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/360ab773-df81-4975-bb19-cd5f34e69b29-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg)

사용자 지정 슬라이싱 전략이 있는 경우, 대신 Circuit에 Barrier를 배치하여 슬라이싱할 위치를 표시하고 [`slice_by_barriers`](../api/qiskit-addon-utils/slicing#slice_by_barriers) 함수를 사용할 수 있습니다.